<a href="https://colab.research.google.com/github/guneetnagia/LearnAI/blob/agentic/AIML27AgenticAI_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#simple Reflex Agent
def simple_reflex(temperature):
  if temperature > 25:
    return "turn AC on "
  else:
    return "turn AC off"

In [ ]:
simple_reflex(25)

'turn AC off'

In [ ]:
#Model based reflex
ac_is_on = False

def model_based_reflex(temperature):
  global ac_is_on
  if temperature > 25 and not ac_is_on:
    return "turn AC on "
  elif temperature < 25 and ac_is_on:
    return "turn AC off"
  else:
    ac_is_on = True
  return "do nothing"

In [ ]:
model_based_reflex(26)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
#Goal based
from openai import OpenAI
client = OpenAI()
goal = 22
def llm_thermostat(temperature):
  response = client.responses.create(
    model="gpt-5-mini",
    input = f"Room is {temperature} C ,Goal is {goal} C"
            f"Choose One action :heat,cool or do nothing .reply with just the action"
  )
  return response.output_text

print(llm_thermostat(26))

cool


In [ ]:
import os
os.environ["OPENWEATHER_API_KEY"] = "4873b42700c0f7f0b8150e70d4161a71"

In [ ]:
!pip install langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI
MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model = MODEL)

In [ ]:
import requests
def fetch_weather(city):
  """Ras call to OpenWeather ,Returns a small claen dictionary"""
  api_key = os.environ["OPENWEATHER_API_KEY"]
  url = "https://api.openweathermap.org/data/2.5/weather"
  r = requests.get(url,params={"q":city,"appid":api_key,"units":"metric"},timeout=10)
  r.raise_for_status()
  d = r.json()
  return {
      "city":d["name"],
    "temperature":d["main"]["temp"],
    "feels_like":d["main"]["feels_like"],
    "humidity":d["main"]["humidity"],
    "condition":d["weather"][0]["description"]
  }

In [ ]:
fetch_weather("hyderabad")

{'city': 'Hyderabad',
 'temperature': 30.71,
 'feels_like': 32.33,
 'humidity': 51,
 'condition': 'broken clouds'}

In [ ]:
from langchain.tools import tool
@tool
def get_weather(city:str)->dict:
  """Get the CURRENT weather for a city,Returns temperature in (C),feels like ,hunidity % condition"""
  return fetch_weather(city)
print(get_weather.name)

get_weather


In [ ]:
print(get_weather.description)

Get the CURRENT weather for a city,Returns temperature in (C),feels like ,hunidity % condition


In [ ]:
print(get_weather.args)

{'city': {'title': 'City', 'type': 'string'}}


In [ ]:

#Agent A
def simple_reflex_agent(city:str) ->str:
  percept = fetch_weather(city)
  #conditions
  if percept["condition"] in ("Rain","Drizzle","Thunderstomr"):
    action ="Carry a umbrella"
  elif  percept["temperature"] >=35:
    action ="Carry water and cap"
  else:
    action="Pleasant weather "
  return f"{percept['city']}:{percept['temperature']}"

In [ ]:
for c in ["Hyderabad","Mumbai","Chennai","Delhi"]:
  print(simple_reflex_agent(c))

Hyderabad:30.71
Mumbai:29.71
Chennai:28.09
Delhi:26.16


In [ ]:
#AGENT B  it keeps internal state
class WeatherWatcher:
  """Model-based reflex agent :remmebers the last percept and reactts to it"""
  def __init__(self,city:str):
    self.city = city
    self.state = None

  def step(self) ->str:
    now = fetch_weather(self.city)
    if self.state in None:
      msg = f"[init]{self.city} :{now['temperature']} C"
    else:
      delta = now["temperature"] - self.state["temperature"]
      if now["condition"] !=self.state["condition"]


In [ ]:
!pip install -q "langchain >=1.0" langchain-openai langgraph

In [ ]:
#AGENTC llm brain agent tool callig with REACT
from datetime import datetime
from langchain.agents import create_agent

@tool
def celcius_to_fahrenheit(celsius:float)->float:
  """Conver ta temperature from celcius to fahrenheit ."""
  return round(celsius *9/5 +32,1)
@tool
def current_time() ->str:
  """Return the current local date and time as string."""
  return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
weather_agent = create_agent(
    model = llm,
    tools = [get_weather,celcius_to_fahrenheit,current_time],
    system_prompt = "You are a concise weather assistant for Indian cities .Use tools for any live fact ,Answer 2-3 lines",


)


In [ ]:
def ask(agent,question:str,config=None,show_trace=True):
  """Run an agent and print the full REACT trcae then the final answer"""
  result = agent.invoke({"messages":[{"role":"user","content":question}]},config=config)
  if show_trace:
    for m in result["messages"]:
      m.pretty_print()
  return result["messages"][-1].content

In [ ]:
ask(weather_agent,"What is the temperature now in hyderabad ?")

================================ Human Message =================================

What is the temperature now in hyderabad ?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_eUMsqyZjkUqbD61nDOnXT6x7)
 Call ID: call_eUMsqyZjkUqbD61nDOnXT6x7
  Args:
    city: Hyderabad
================================= Tool Message =================================
Name: get_weather

{"city": "Hyderabad", "temperature": 30.55, "feels_like": 32.07, "humidity": 51, "condition": "broken clouds"}
================================== Ai Message ==================================

The current temperature in Hyderabad is 30.5°C, with a feels-like temperature of 32.1°C. The humidity level is at 51%, and the weather condition is broken clouds.


'The current temperature in Hyderabad is 30.5°C, with a feels-like temperature of 32.1°C. The humidity level is at 51%, and the weather condition is broken clouds.'

In [ ]:
ask(weather_agent,"Is bengaluru warmer than Delhi right now?Give both in Fahrenheit and tell me what time it is.")

================================ Human Message =================================

Is bengaluru warmer than Delhi right now?Give both in Fahrenheit and tell me what time it is.
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_Uo9CqUmYqtegepy2Oii9UCWm)
 Call ID: call_Uo9CqUmYqtegepy2Oii9UCWm
  Args:
    city: Bengaluru
  get_weather (call_Y0RUZ6hd74jeMfB1FwtPq5cq)
 Call ID: call_Y0RUZ6hd74jeMfB1FwtPq5cq
  Args:
    city: Delhi
  current_time (call_Sw75ryTaUCV9iWDHdH0ZJBPz)
 Call ID: call_Sw75ryTaUCV9iWDHdH0ZJBPz
  Args:
================================= Tool Message =================================
Name: get_weather

{"city": "Bengaluru", "temperature": 27.23, "feels_like": 29.64, "humidity": 74, "condition": "light rain"}
================================= Tool Message =================================
Name: get_weather

{"city": "Delhi", "temperature": 27.52, "feels_like": 27.93, "humidity": 50, "condition": "clear sky"}
=

'Bengaluru is currently 81.0°F (27.23°C) while Delhi is 81.5°F (27.52°C). Therefore, Bengaluru is slightly cooler than Delhi. The current local time is 05:35 AM.'

In [ ]:
#Agent Type D -Conversational agent with memory
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
chat_agent = create_agent(
    model=llm,
    tools=[get_weather,celcius_to_fahrenheit],
    system_prompt="You are a friendly weather assistant.Remember facts athe user tells you about themselves.",
    checkpointer=InMemorySaver()

)

In [ ]:
habeeb = {"configurable":{"thread_id":"user-habeeb"}}
print(ask(chat_agent,'Hi I live im hyderabad and i jog everymoorning',config=habeeb,show_trace=False))

That's great to hear! Jogging every morning is a fantastic way to stay fit. How can I assist you with your morning jog in Hyderabad? Would you like to know the weather forecast for today?


In [ ]:
print(ask(chat_agent,"Is it ok to go now",config=habeeb,show_trace=False))

The current temperature in Hyderabad is about 32.1°C, feeling like 32.8°C, with a humidity of 42%. The weather condition is "broken clouds." 

It seems like a relatively warm and humid day for a jog. Make sure to stay hydrated and wear appropriate attire. If you enjoy jogging in such weather, it should be okay to go out now!


In [ ]:
print(ask(chat_agent,"what did i say my hobby was?",config=habeeb,show_trace=False))

You mentioned that you jog every morning. If you have any other hobbies or interests you'd like to share, feel free to let me know!


In [ ]:
priya = {"configurable":{"thread_id":"user-priya"}}
print(ask(chat_agent,"is it okay to go now?",config=priya,show_trace=False))

Could you please let me know where you are planning to go? I can check the weather for that location to help you decide!


In [ ]:
#Goal based agent
#pattern given a goal,plan,executes,synthesis
from pydantic import BaseModel,Field

In [ ]:
#use case plan a weekend trip for my team choose between hyderabad,bengalur and pune

In [ ]:
class Plan(BaseModel):
  goal:str=Field(description="restate the gaol in one line")
  cities_to_check:list[str] =Field(description="Cities whose weather must be fetched")
  decision_rule:str = Field(description="How the finak choice will be made ,in one sentence")

In [ ]:
planner = llm.with_structured_output(Plan)

In [ ]:
#PLANB
goal = "Plan a Saturday afternoon team picnic.Candiate cities:Hyderabad,Bengaluru,Pune.Pick the best one"
plan = planner.invoke(f"Create a plan to achieve this goal using a get_weather(city) tool:{goal}")
print("Plan",plan.model_dump_json(indent=2))

Plan {
  "goal": "Plan a Saturday afternoon team picnic by choosing the best city based on weather.",
  "cities_to_check": [
    "Hyderabad",
    "Bengaluru",
    "Pune"
  ],
  "decision_rule": "The best city will be selected based on the most favorable weather conditions for a picnic."
}


In [ ]:
#execut Each step
observations = {c:fetch_weather(c) for c in plan.cities_to_check}
print(observations)
for c,w in observations.items():
  print(f"{c}:{w['temperature']} C")
#

{'Hyderabad': {'city': 'Hyderabad', 'temperature': 32.11, 'feels_like': 32.81, 'humidity': 42, 'condition': 'broken clouds'}, 'Bengaluru': {'city': 'Bengaluru', 'temperature': 27.23, 'feels_like': 29.44, 'humidity': 72, 'condition': 'overcast clouds'}, 'Pune': {'city': 'Pune', 'temperature': 32.61, 'feels_like': 31.28, 'humidity': 28, 'condition': 'overcast clouds'}}
Hyderabad:32.11 C
Bengaluru:27.23 C
Pune:32.61 C


In [ ]:
final = llm.invoke(
    f"Goal:{plan.goal} \n decision rule :{plan.decision_rule}\n observations{observations}\n"
    "Give the recommendation in 3 lines with reason"
)
print(final.content)

Based on the weather conditions, I recommend holding the picnic in Pune. It has the warmest temperature at 32.61°C with low humidity at 28%, which is more comfortable for outdoor activities. While Hyderabad also has favorable conditions, Pune's lower humidity will enhance the overall picnic experience.


In [ ]:
#utility based agent

In [ ]:
#rank five citites by an "out door comfort" score for this evening

In [ ]:
def comfort_score(w:dict)->float:
  raining = w["condition"] in ("Rain","Drizzle","Thunderstorm")
  score = 100 - 2 * abs(w["temperature"]-26)-0.5*max(w["humidity"]-50,0)-40* raining
  return round(score,1)

In [ ]:
def utility_agent(cities:list[str])->str:
  scored = []
  for c in cities:
    w = fetch_weather(c)
    scored.append((comfort_score(w),c,w))
  scored.sort(reverse=True)
  table = "\n".join(f"{s:>5} {c:<10} {w['temperature']} {w['humidity']}" for s,c,w in scored)
  explanation=llm.invoke(
      f"Here are cities ranked by an outdoor comfort score (higher is the better):\n{table}\n"
       "In 3 lines ,tell a non technical manager which city to pick for an evening event and why,do not invent numbers"
  ).content
  return table+"\n\n"+explanation

In [ ]:
print(utility_agent(["Hyderabad","Bengaluru","Pune","chennai"]))

 88.4 Pune       31.81 27
 88.2 Hyderabad  31.88 42
 85.7 Bengaluru  27.67 72
 80.6 chennai    29.45 75

I recommend choosing Pune for the evening event, as it has the highest outdoor comfort score of 88.4, making it the most pleasant environment for attendees. Hyderabad is a close second at 88.2, but Pune's edge in comfort can enhance the overall experience. Choosing a more comfortable city will likely lead to better engagement and enjoyment from guests.
